## SRP135786

**paper:** [PMID: 31660689](https://onlinelibrary.wiley.com/doi/full/10.1111/1755-0998.13110?saml_referrer) - The utility of reptile blood transcriptomes in molecular ecology, 2020

**date, curator:** 2026-04-20, Sara Carsanaro

**notes**

### annotation summary

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Whole Blood,UBERON:0000178,blood,perfect match


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,Adult,UBERON:0000113,post-juvenile adult stage,perfect match
1,Juvenile,UBERON:0034919,juvenile stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP135786"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████████| 6/6 [00:05<00:00,  1.04it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX3797549,SRP135786,Illumina HiSeq 2500,SRS3049607,,,,,,Whole Blood,Adult,,,,M,,,94850,,,,,,Pcat,SAMN08718444,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Pcat,,,,,,TRANSCRIPTOMIC,Oligo-dT
1,SRX3797548,SRP135786,Illumina HiSeq 2500,SRS3049606,,,,,,Whole Blood,Juvenile,,,,F,,,35005,,,,,,Tele,SAMN08718445,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Tele,,,,,,TRANSCRIPTOMIC,Oligo-dT
2,SRX3797547,SRP135786,Illumina HiSeq 2500,SRS3049605,,,,,,Whole Blood,Adult,,,,NA,,,71008,,,,,,Emul,SAMN08718442,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Emul,,,,,,TRANSCRIPTOMIC,Oligo-dT
3,SRX3797546,SRP135786,Illumina HiSeq 2500,SRS3049603,,,,,,Whole Blood,Adult,,,,M,,,8519,,,,,,Socc,SAMN08718443,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Socc,,,,,,TRANSCRIPTOMIC,Oligo-dT
4,SRX3797545,SRP135786,Illumina HiSeq 2500,SRS3049604,,,,,,Whole Blood,Juvenile,,,,NA,,,8479,,,,,,Cpic,SAMN08718446,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Cpic,,,,,,TRANSCRIPTOMIC,Oligo-dT
5,SRX3797544,SRP135786,Illumina HiSeq 2500,SRS3049602,,,,,,Whole Blood,Adult,,,,F,,,8590,,,,,,Ccon,SAMN08718447,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Ccon,,,,,,TRANSCRIPTOMIC,Oligo-dT


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Whole Blood']


In [6]:

# all
library.loc[:,'anatId'] = 'UBERON:0000178'
library.loc[:,'anatName'] = 'blood'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX3797549,SRP135786,Illumina HiSeq 2500,SRS3049607,UBERON:0000178,blood,,,,Whole Blood,Adult,perfect match,not documented,,M,,,94850,,,,,,Pcat,SAMN08718444,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Pcat,,,,,,TRANSCRIPTOMIC,Oligo-dT
1,SRX3797548,SRP135786,Illumina HiSeq 2500,SRS3049606,UBERON:0000178,blood,,,,Whole Blood,Juvenile,perfect match,not documented,,F,,,35005,,,,,,Tele,SAMN08718445,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Tele,,,,,,TRANSCRIPTOMIC,Oligo-dT
2,SRX3797547,SRP135786,Illumina HiSeq 2500,SRS3049605,UBERON:0000178,blood,,,,Whole Blood,Adult,perfect match,not documented,,NA,,,71008,,,,,,Emul,SAMN08718442,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Emul,,,,,,TRANSCRIPTOMIC,Oligo-dT
3,SRX3797546,SRP135786,Illumina HiSeq 2500,SRS3049603,UBERON:0000178,blood,,,,Whole Blood,Adult,perfect match,not documented,,M,,,8519,,,,,,Socc,SAMN08718443,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Socc,,,,,,TRANSCRIPTOMIC,Oligo-dT
4,SRX3797545,SRP135786,Illumina HiSeq 2500,SRS3049604,UBERON:0000178,blood,,,,Whole Blood,Juvenile,perfect match,not documented,,NA,,,8479,,,,,,Cpic,SAMN08718446,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Cpic,,,,,,TRANSCRIPTOMIC,Oligo-dT
5,SRX3797544,SRP135786,Illumina HiSeq 2500,SRS3049602,UBERON:0000178,blood,,,,Whole Blood,Adult,perfect match,not documented,,F,,,8590,,,,,,Ccon,SAMN08718447,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Ccon,,,,,,TRANSCRIPTOMIC,Oligo-dT


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['Adult' 'Juvenile']


In [8]:
# conditional (based off infoStage)
library.loc[library["infoStage"] == "Adult", "stageId"] = "UBERON:0000113"
library.loc[library["infoStage"] == "Adult", "stageName"] = "post-juvenile adult stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "Adult", "stageAnnotationStatus"] = "perfect match"

# conditional (based off infoStage)
library.loc[library["infoStage"] == "Juvenile", "stageId"] = "UBERON:0034919"
library.loc[library["infoStage"] == "Juvenile", "stageName"] = "juvenile stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "Juvenile", "stageAnnotationStatus"] = "perfect match"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX3797549,SRP135786,Illumina HiSeq 2500,SRS3049607,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,M,,,94850,,,,,,Pcat,SAMN08718444,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Pcat,,,,,,TRANSCRIPTOMIC,Oligo-dT
1,SRX3797548,SRP135786,Illumina HiSeq 2500,SRS3049606,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Whole Blood,Juvenile,perfect match,not documented,perfect match,F,,,35005,,,,,,Tele,SAMN08718445,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Tele,,,,,,TRANSCRIPTOMIC,Oligo-dT
2,SRX3797547,SRP135786,Illumina HiSeq 2500,SRS3049605,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,NA,,,71008,,,,,,Emul,SAMN08718442,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Emul,,,,,,TRANSCRIPTOMIC,Oligo-dT
3,SRX3797546,SRP135786,Illumina HiSeq 2500,SRS3049603,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,M,,,8519,,,,,,Socc,SAMN08718443,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Socc,,,,,,TRANSCRIPTOMIC,Oligo-dT
4,SRX3797545,SRP135786,Illumina HiSeq 2500,SRS3049604,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Whole Blood,Juvenile,perfect match,not documented,perfect match,NA,,,8479,,,,,,Cpic,SAMN08718446,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Cpic,,,,,,TRANSCRIPTOMIC,Oligo-dT
5,SRX3797544,SRP135786,Illumina HiSeq 2500,SRS3049602,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,F,,,8590,,,,,,Ccon,SAMN08718447,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Ccon,,,,,,TRANSCRIPTOMIC,Oligo-dT


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'SureSelect Strand-Specific RNA Library Preparation Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX3797549,SRP135786,Illumina HiSeq 2500,SRS3049607,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,M,,,94850,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Pcat,SAMN08718444,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Pcat,,,,,,TRANSCRIPTOMIC,Oligo-dT
1,SRX3797548,SRP135786,Illumina HiSeq 2500,SRS3049606,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Whole Blood,Juvenile,perfect match,not documented,perfect match,F,,,35005,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Tele,SAMN08718445,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Tele,,,,,,TRANSCRIPTOMIC,Oligo-dT
2,SRX3797547,SRP135786,Illumina HiSeq 2500,SRS3049605,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,NA,,,71008,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Emul,SAMN08718442,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Emul,,,,,,TRANSCRIPTOMIC,Oligo-dT
3,SRX3797546,SRP135786,Illumina HiSeq 2500,SRS3049603,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,M,,,8519,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Socc,SAMN08718443,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Socc,,,,,,TRANSCRIPTOMIC,Oligo-dT
4,SRX3797545,SRP135786,Illumina HiSeq 2500,SRS3049604,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Whole Blood,Juvenile,perfect match,not documented,perfect match,NA,,,8479,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Cpic,SAMN08718446,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Cpic,,,,,,TRANSCRIPTOMIC,Oligo-dT
5,SRX3797544,SRP135786,Illumina HiSeq 2500,SRS3049602,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,F,,,8590,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Ccon,SAMN08718447,,,,,,,,,,,30/04/2026,Agilent SureSelect Stranded library preparation kit,,Ccon,,,,,,TRANSCRIPTOMIC,Oligo-dT


#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-30'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX3797549,SRP135786,Illumina HiSeq 2500,SRS3049607,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,M,,,94850,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Pcat,SAMN08718444,,,,,,,,,,SAC,2026-03-30,Agilent SureSelect Stranded library preparation kit,,Pcat,,,,,,TRANSCRIPTOMIC,Oligo-dT
1,SRX3797548,SRP135786,Illumina HiSeq 2500,SRS3049606,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Whole Blood,Juvenile,perfect match,not documented,perfect match,F,,,35005,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Tele,SAMN08718445,,,,,,,,,,SAC,2026-03-30,Agilent SureSelect Stranded library preparation kit,,Tele,,,,,,TRANSCRIPTOMIC,Oligo-dT
2,SRX3797547,SRP135786,Illumina HiSeq 2500,SRS3049605,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,NA,,,71008,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Emul,SAMN08718442,,,,,,,,,,SAC,2026-03-30,Agilent SureSelect Stranded library preparation kit,,Emul,,,,,,TRANSCRIPTOMIC,Oligo-dT
3,SRX3797546,SRP135786,Illumina HiSeq 2500,SRS3049603,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,M,,,8519,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Socc,SAMN08718443,,,,,,,,,,SAC,2026-03-30,Agilent SureSelect Stranded library preparation kit,,Socc,,,,,,TRANSCRIPTOMIC,Oligo-dT
4,SRX3797545,SRP135786,Illumina HiSeq 2500,SRS3049604,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Whole Blood,Juvenile,perfect match,not documented,perfect match,NA,,,8479,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Cpic,SAMN08718446,,,,,,,,,,SAC,2026-03-30,Agilent SureSelect Stranded library preparation kit,,Cpic,,,,,,TRANSCRIPTOMIC,Oligo-dT
5,SRX3797544,SRP135786,Illumina HiSeq 2500,SRS3049602,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,F,,,8590,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Ccon,SAMN08718447,,,,,,,,,,SAC,2026-03-30,Agilent SureSelect Stranded library preparation kit,,Ccon,,,,,,TRANSCRIPTOMIC,Oligo-dT


#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 31660689'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX3797549,SRP135786,Illumina HiSeq 2500,SRS3049607,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,M,,,94850,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Pcat,SAMN08718444,,,,,,,PMID: 31660689,,,SAC,2026-03-30
1,SRX3797548,SRP135786,Illumina HiSeq 2500,SRS3049606,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Whole Blood,Juvenile,perfect match,not documented,perfect match,F,,,35005,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Tele,SAMN08718445,,,,,,,PMID: 31660689,,,SAC,2026-03-30
2,SRX3797547,SRP135786,Illumina HiSeq 2500,SRS3049605,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,NA,,,71008,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Emul,SAMN08718442,,,,,,,PMID: 31660689,,,SAC,2026-03-30
3,SRX3797546,SRP135786,Illumina HiSeq 2500,SRS3049603,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,M,,,8519,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Socc,SAMN08718443,,,,,,,PMID: 31660689,,,SAC,2026-03-30
4,SRX3797545,SRP135786,Illumina HiSeq 2500,SRS3049604,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Whole Blood,Juvenile,perfect match,not documented,perfect match,NA,,,8479,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Cpic,SAMN08718446,,,,,,,PMID: 31660689,,,SAC,2026-03-30
5,SRX3797544,SRP135786,Illumina HiSeq 2500,SRS3049602,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,F,,,8590,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,polyA,,,Ccon,SAMN08718447,,,,,,,PMID: 31660689,,,SAC,2026-03-30


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP135786,Reptile Blood Transcriptomes,"The goal of this project was to gain a general understanding of what genes and genetic networks are transcribed in retile blood cells. Reptiles have nucleated red blood cells along with the white blood cells. This may allow repeated sampling of blood to study temporal transcriptomics responses to environmental stressors, rather than euthanizing the animals to assay other tissues.",SRA,,,,,,,PRJNA438493,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

6

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP135786,Reptile Blood Transcriptomes,"The goal of this project was to gain a general understanding of what genes and genetic networks are transcribed in retile blood cells. Reptiles have nucleated red blood cells along with the white blood cells. This may allow repeated sampling of blood to study temporal transcriptomics responses to environmental stressors, rather than euthanizing the animals to assay other tissues.",SRA,total,Bgee 1K,6,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,,PRJNA438493,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '31660689'
experiment.loc[:,'reference_url'] = 'https://onlinelibrary.wiley.com/doi/full/10.1111/1755-0998.13110?saml_referrer'
experiment.loc[:,'DOI'] = '10.1111/1755-0998.13110'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP135786,Reptile Blood Transcriptomes,"The goal of this project was to gain a general understanding of what genes and genetic networks are transcribed in retile blood cells. Reptiles have nucleated red blood cells along with the white blood cells. This may allow repeated sampling of blood to study temporal transcriptomics responses to environmental stressors, rather than euthanizing the animals to assay other tissues.",SRA,total,Bgee 1K,6,SureSelect Strand-Specific RNA Library Preparation Kit,full_length,,PRJNA438493,31660689,https://onlinelibrary.wiley.com/doi/full/10.1111/1755-0998.13110?saml_referrer,10.1111/1755-0998.13110,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [20]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
62696,SRX2549119,SRP099173,Illumina HiSeq 2000,SRS1967582,UBERON:0004122,genitourinary system,UBERON:0000111,organogenesis stage,,adrenal-kidney-gonad complex,CPI stage 12,other,not documented,other,M,,,8479,,,polyA,,,CPI_26_12,"SAMN06298504,GSM2486782",,,,,,,PMID: 28296881,26 C,,SAC,2026-04-30
62697,SRX2549120,SRP099173,Illumina HiSeq 2000,SRS1967581,UBERON:0002100,trunk,UBERON:0000111,organogenesis stage,,trunks,CPI stage 9,perfect match,not documented,other,M,,,8479,,,polyA,,,CPI_26_9,"SAMN06298503,GSM2486781",,,,,,,PMID: 28296881,26 C,,SAC,2026-04-30
62698,SRX3797549,SRP135786,Illumina HiSeq 2500,SRS3049607,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,M,,,94850,SureSelect Strand-Specific RNA Library Prepara...,full_length,polyA,,,Pcat,SAMN08718444,,,,,,,PMID: 31660689,,,SAC,2026-03-30
62699,SRX3797548,SRP135786,Illumina HiSeq 2500,SRS3049606,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Whole Blood,Juvenile,perfect match,not documented,perfect match,F,,,35005,SureSelect Strand-Specific RNA Library Prepara...,full_length,polyA,,,Tele,SAMN08718445,,,,,,,PMID: 31660689,,,SAC,2026-03-30
62700,SRX3797547,SRP135786,Illumina HiSeq 2500,SRS3049605,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,NA,,,71008,SureSelect Strand-Specific RNA Library Prepara...,full_length,polyA,,,Emul,SAMN08718442,,,,,,,PMID: 31660689,,,SAC,2026-03-30
62701,SRX3797546,SRP135786,Illumina HiSeq 2500,SRS3049603,UBERON:0000178,blood,UBERON:0000113,post-juvenile adult stage,,Whole Blood,Adult,perfect match,not documented,perfect match,M,,,8519,SureSelect Strand-Specific RNA Library Prepara...,full_length,polyA,,,Socc,SAMN08718443,,,,,,,PMID: 31660689,,,SAC,2026-03-30
62702,SRX3797545,SRP135786,Illumina HiSeq 2500,SRS3049604,UBERON:0000178,blood,UBERON:0034919,juvenile stage,,Whole Blood,Juvenile,perfect match,not documented,perfect match,NA,,,8479,SureSelect Strand-Specific RNA Library Prepara...,full_length,polyA,,,Cpic,SAMN08718446,,,,,,,PMID: 31660689,,,SAC,2026-03-30


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1211,SRP077056,Gene network variation and alternative paths t...,Diversification of the turtle's shell comprise...,SRA,total,Bgee 1K,30,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA323738,30094964,https://onlinelibrary.wiley.com/doi/epdf/10.11...,10.1111/ede.12264,,
1212,SRP099173,Time series embryonic gonadal transcriptome of...,Study of embryonic gonadal transcriptome acros...,SRA,total,Bgee 1K,20,,,GSE94871,PRJNA371383,28296881,https://pmc.ncbi.nlm.nih.gov/articles/PMC5352168/,10.1371/journal.pone.0172044,,
1213,SRP135786,Reptile Blood Transcriptomes,The goal of this project was to gain a general...,SRA,total,Bgee 1K,6,SureSelect Strand-Specific RNA Library Prepara...,full_length,,PRJNA438493,31660689,https://onlinelibrary.wiley.com/doi/full/10.11...,10.1111/1755-0998.13110,,


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop 7f93d8c] adding annotated bulk experiment SRP135786
 2 files changed, 7 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.19 KiB | 2.19 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   1c72850..7f93d8c  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push